In [2]:
from datasets import load_dataset, load_from_disk
import polars as pl
import pickle
import scipy
import os
import pandas as pd
from tqdm import tqdm
import numpy as np
import datasets
import os
from datetime import datetime

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

DATA_PATH = "/home/jupyter/filestore/storage/datasets"

In [3]:
dataset = load_from_disk(f"{DATA_PATH}/user_events_20230501")
polars_ds = dataset.to_polars()

In [4]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

hist_len = 400

train_items = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
    .select(
        pl.col("user_id"),
        pl.col("item_id"),
        pl.col("name"),
        pl.col("stime"),
        pl.col("stime").rank("dense", descending=True).over("user_id").alias("rn")
    )
    .filter(pl.col("rn") <= hist_len)
    .select("item_id", "name")
    .unique()
)

In [6]:
train_items.head(5)

item_id,name
i64,str
167505134,"""Custom Batman blue leather wired cape Mcfarlane Mezco one:12 NOT FIGURE"""
174340789,"""Nike Dunks mid"""
46050787,"""Lenovo Thinkplus TH10 Wireless Sport Headphones Waterproof Headset"""
57735798,"""Hello kitty iPad case, IPAD PRO 11 2020"""
102394042,"""Juicy Couture pink velour pijama set XL"""


In [5]:
item_names = train_items["name"].to_list()
item_ids = train_items["item_id"].to_list()

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 653.04it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
embeddings = model.encode(item_names, batch_size=4096, show_progress_bar=True, normalize_embeddings=True)

Batches: 100%|██████████| 985/985 [05:10<00:00,  3.18it/s]


In [8]:
item2id = {item: ids for ids, item in enumerate(item_ids)}

In [9]:
with open(f'item2id_hist_len{hist_len}.pickle', 'wb') as f:
    pickle.dump(item2id, f)
    
np.save(f"item_embs_hist_len{hist_len}", embeddings)